# Fine-tune OpenTSLM on Amazon SageMaker

Fine-tune the pretrained OpenTSLM model ([opentime/tslm-nov26](https://huggingface.co/opentime/tslm-nov26) on Hugging Face) on ARFBench as a SageMaker training job, then deploy the fine-tuned model to a real-time endpoint and ask it about a time series.

Run it on your machine with AWS credentials for the account and region to use (for example after `aws sso login`, with `AWS_PROFILE` and `AWS_DEFAULT_REGION` set), or in SageMaker Studio.

Keep the `scripts` folder next to this notebook. `scripts/utils.py` holds the OpenTSLM model, its fine-tuning code and the endpoint's request handler, which the training job and the endpoint both import. `scripts/train.py` is the training job's entry point.

Fine-tuning runs on one `ml.g7e.2xlarge` instance and the endpoint on one `ml.g6.2xlarge`. We ran it in eu-north-1; check that your region offers both types. New accounts often have a quota of 0 for both. Before you start, request at least 1 for "ml.g7e.2xlarge for training job usage" and "ml.g6.2xlarge for endpoint usage" in Service Quotas.

## Set up

The next cell finds or creates what the notebook needs in your account: the SageMaker default bucket (`sagemaker-<region>-<account ID>`) and an execution role. In SageMaker Studio it uses the Studio execution role. Elsewhere it creates the role `OpenTSLMSageMakerRole` once, with the AWS managed policy `AmazonSageMakerFullAccess`, so your credentials need permission to create IAM roles the first time.

In [ ]:
%pip install --quiet "sagemaker>=3.29,<4" "botocore[crt]"  # crt: for profiles that use aws login

In [ ]:
import json
import os
import time
import uuid
from pathlib import Path

from sagemaker.core.helper.session_helper import Session, get_execution_role

session = Session()
REGION = session.boto_region_name
BUCKET = session.default_bucket()  # created if it does not exist


def execution_role(name="OpenTSLMSageMakerRole"):
    if os.path.exists("/opt/ml/metadata/resource-metadata.json"):  # SageMaker Studio
        return get_execution_role(session)
    iam = session.boto_session.client("iam")
    try:
        return iam.get_role(RoleName=name)["Role"]["Arn"]
    except iam.exceptions.NoSuchEntityException:
        pass
    trust = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {"Service": "sagemaker.amazonaws.com"},
                "Action": "sts:AssumeRole",
            }
        ],
    }
    arn = iam.create_role(RoleName=name, AssumeRolePolicyDocument=json.dumps(trust))["Role"]["Arn"]
    iam.attach_role_policy(
        RoleName=name, PolicyArn="arn:aws:iam::aws:policy/AmazonSageMakerFullAccess"
    )
    time.sleep(15)  # SageMaker can assume a new role only after a few seconds
    return arn


ROLE_ARN = execution_role()
unique_id = uuid.uuid4().hex[:8]
finetuned_endpoint_name = f"opentslm-arfbench-{unique_id}"
print(f"Region {REGION}, bucket {BUCKET}, role {ROLE_ARN}")

## Fine-tune on ARFBench

ARFBench is a public benchmark of multiple-choice questions about anomalies in Datadog's software-incident telemetry. TimeNet serves it as a versioned dataset of tasks: each question with its options, its answer, and the metric series it refers to.

`scripts/arfbench-split.json` assigns each question to the training, validation or test set: about 70%, 10% and 20% of the questions. All questions about one incident are in the same set, so the model is never tested on an incident it trained on. The answers of one incident are often alike, so a random split can put most questions with one answer in one set. `split_by_incident` in `utils.py` therefore searches for a split where each set has about the same mix of question categories and answers. It made the file with `save_split(split_by_incident(dataset.tasks), "arfbench-split.json")`; every run reads the same file, so all runs train, choose the best epoch, and test on the same questions.

Each category is also scored by its balanced accuracy, the mean share right per correct answer. Most presence questions have an anomaly, so a model that always answers "yes" gets a high accuracy there, but only 50% balanced accuracy.

`ModelTrainer` runs `scripts/train.py` as a SageMaker training job on one GPU. It uploads the `scripts` folder, installs `scripts/requirements.txt` in the SageMaker PyTorch 2.9 training image, and passes the settings as hyperparameters. The job writes the fine-tuned model and `metrics.json`, the test results before and after fine-tuning, to `FINETUNED_PREFIX`.

Optionally, the job also logs its parameters, the loss of each epoch and the test results to an [MLflow App](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow-app-setup.html). With `MLFLOW_APP = "default"`, the next cell uses the default MLflow App of your SageMaker Studio domain, or the default MLflow App of your account. Set an MLflow App ARN to use that app, or `None` to turn MLflow off. The execution role needs permission to call the app (`sagemaker:CallMlflowAppApi`, which `AmazonSageMakerFullAccess` includes), and the `mlflow-skinny` version in `scripts/requirements.txt` should match the MLflow version of the app.

In [ ]:
from botocore.exceptions import ClientError

MLFLOW_APP = "default"  # "default", an MLflow App ARN, or None


def mlflow_app_arn(spec):
    """Give the ARN of the MLflow App to log to, or None."""
    if spec != "default":
        return spec
    sagemaker_client = session.boto_session.client("sagemaker")
    metadata = Path("/opt/ml/metadata/resource-metadata.json")  # SageMaker Studio
    domain = json.loads(metadata.read_text()).get("DomainId") if metadata.exists() else None
    try:
        apps = [
            sagemaker_client.describe_mlflow_app(Arn=app["Arn"])
            for page in sagemaker_client.get_paginator("list_mlflow_apps").paginate()
            for app in page["Summaries"]
            if app["Status"] in ("Created", "Updated")
        ]
    except ClientError as error:
        print(f"Cannot list the MLflow Apps: {error}")
        return None
    for app in apps:  # the default app of this Studio domain
        if domain and domain in app.get("DefaultDomainIdList", []):
            return app["Arn"]
    for app in apps:  # the default app of the account
        if app.get("AccountDefaultStatus") == "ENABLED":
            return app["Arn"]
    return None


MLFLOW_APP_ARN = mlflow_app_arn(MLFLOW_APP)
print(f"MLflow App: {MLFLOW_APP_ARN or 'none, the job logs only to CloudWatch'}")


In [ ]:
from sagemaker.core import image_uris
from sagemaker.core.s3 import S3Downloader
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import Compute, SourceCode, StoppingCondition

# The pretrained model to fine-tune: a Hugging Face repository (hf://ORG/REPO) or a local model
# directory. The repository can get newer checkpoints over time, so set this to the checkpoint
# that fits your use case.
BASE_MODEL = "hf://opentime/tslm-nov26"
FINETUNED_PREFIX = f"s3://{BUCKET}/opentslm-arfbench/{unique_id}"
# 1 x NVIDIA RTX PRO 6000 (96 GB). ml.g7e.4xlarge and ml.g7e.8xlarge have the same GPU: use one
# of them if the job waits for capacity. Fine-tuning needs most of the 96 GB.
TRAINING_INSTANCE_TYPE = "ml.g7e.2xlarge"

environment = {"HF_HOME": "/opt/ml/input/huggingface"}
if MLFLOW_APP_ARN:
    environment["MLFLOW_TRACKING_ARN"] = MLFLOW_APP_ARN

trainer = ModelTrainer(
    training_image=image_uris.retrieve(
        "pytorch",
        region=REGION,
        version="2.9.0",
        py_version="py312",
        instance_type=TRAINING_INSTANCE_TYPE,
        image_scope="training",
    ),
    source_code=SourceCode(
        source_dir="scripts", entry_script="train.py", requirements="requirements.txt"
    ),
    compute=Compute(instance_type=TRAINING_INSTANCE_TYPE, volume_size_in_gb=100),
    # ModelTrainer stops a job after 1 hour by default; fine-tuning can take longer.
    stopping_condition=StoppingCondition(max_runtime_in_seconds=24 * 60 * 60),
    hyperparameters={"base-model": BASE_MODEL, "output": FINETUNED_PREFIX, "epochs": 3},
    environment=environment,
    role=ROLE_ARN,
    sagemaker_session=session,
    base_job_name="opentslm-arfbench",
)
trainer.train()  # waits for the job and shows its log

results = json.loads(
    S3Downloader.read_file(f"{FINETUNED_PREFIX}/metrics.json", sagemaker_session=session)
)
before, after = results["before"], results["after"]
print(f"Test accuracy before fine-tuning: {before['accuracy']:.1%}")
print(f"Test accuracy after fine-tuning:  {after['accuracy']:.1%}\n")
print(f"{'Category':<26} {'Questions':>9} {'Accuracy':>9} {'Balanced':>9}")
for category, scores in after["per_category"].items():
    print(
        f"{category:<26} {scores['questions']:>9} {scores['accuracy']:>9.1%}"
        f" {scores['balanced_accuracy']:>9.1%}"
    )


## Deploy the fine-tuned model

The SageMaker PyTorch inference image serves an S3 prefix that holds the model files and a `code/` folder. The next cell puts three files in that folder: `scripts/utils.py`, `inference.py`, which takes the request handler from `utils.py`, and `requirements.txt`, which installs Transformers when the endpoint starts. OpenTSLM needs one GPU with at least 24 GB of memory and bfloat16 support, such as `ml.g6.2xlarge` (1 x NVIDIA L4).

A request holds up to 4 questions. Each question has up to 32 series and the question text; a series has its `values` (`None` marks a missing value), a short `name`, and a `text` that describes it. The response holds one answer per question.

In [ ]:
from sagemaker.core.s3 import S3Uploader
from sagemaker.serve.model_builder import ModelBuilder

INFERENCE_INSTANCE_TYPE = "ml.g6.2xlarge"

S3Uploader.upload("scripts/utils.py", f"{FINETUNED_PREFIX}/code", sagemaker_session=session)
for name, body in [
    ("inference.py", "from utils import input_fn, model_fn, output_fn, predict_fn\n"),
    ("requirements.txt", "transformers==4.57.6\n"),  # the image has torch and numpy
]:
    S3Uploader.upload_string_as_file_body(
        body, f"{FINETUNED_PREFIX}/code/{name}", sagemaker_session=session
    )

finetuned_builder = ModelBuilder(
    image_uri=image_uris.retrieve(
        "pytorch",
        region=REGION,
        version="2.6.0",
        py_version="py312",
        instance_type=INFERENCE_INSTANCE_TYPE,
        image_scope="inference",
    ),
    env_vars={
        "SAGEMAKER_PROGRAM": "inference.py",
        "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
        "SAGEMAKER_MODEL_SERVER_WORKERS": "1",  # one copy of the model per GPU
        "TS_DEFAULT_STARTUP_TIMEOUT": "1800",  # loading the model takes several minutes
        "SAGEMAKER_TS_RESPONSE_TIMEOUT": "300",
    },
    s3_model_data_url={
        "S3DataSource": {
            "S3Uri": f"{FINETUNED_PREFIX}/",
            "S3DataType": "S3Prefix",
            "CompressionType": "None",
        }
    },
    role_arn=ROLE_ARN,
    instance_type=INFERENCE_INSTANCE_TYPE,
)
finetuned_model = finetuned_builder.build(model_name=finetuned_endpoint_name)
finetuned_endpoint = finetuned_builder.deploy(
    endpoint_name=finetuned_endpoint_name,
    model_data_download_timeout=1800,  # the weights take 14 GB
    container_startup_health_check_timeout=1800,
)


In [ ]:
def prompt(endpoint, series, question):
    request = {
        "instances": [{"series": series, "question": question}],
        "parameters": {"max_new_tokens": 128},
    }
    response = endpoint.invoke(body=json.dumps(request), content_type="application/json")
    return json.loads(response.body.read())["generated_text"][0]


# One hour of CPU utilization, one sample per minute, with a five-minute spike at minute 28.
cpu = [21, 20, 22, 19] * 7 + [78, 85, 82, 80, 79] + [22, 20, 21] * 9
cpu_series = [
    {"values": cpu, "name": "CPU utilization (%)", "text": "CPU utilization (%), 1 sample/minute:"}
]
question = "Is there an anomaly in this series? If so, when does it start and end?"
print(prompt(finetuned_endpoint, cpu_series, question))

## Clean up

The next cell deletes the endpoint, its configuration and the SageMaker model, also after a failed deploy. The fine-tuned model files and `metrics.json` stay under `FINETUNED_PREFIX` in your bucket, the training job's files stay in your bucket under the job name, and the role `OpenTSLMSageMakerRole` stays for the next run.

In [ ]:
sagemaker_client = session.boto_session.client("sagemaker")
try:
    config = sagemaker_client.describe_endpoint(EndpointName=finetuned_endpoint_name)
    sagemaker_client.delete_endpoint(EndpointName=finetuned_endpoint_name)
    sagemaker_client.delete_endpoint_config(EndpointConfigName=config["EndpointConfigName"])
except sagemaker_client.exceptions.ClientError as error:
    print(error)  # no endpoint: the deploy did not get that far
try:
    sagemaker_client.delete_model(ModelName=finetuned_endpoint_name)
except sagemaker_client.exceptions.ClientError as error:
    print(error)